# 3.2.1 Price Dynamics and Volatility (Light, Load-Only)

This notebook is intentionally **load-only**. It does not simulate, calibrate, or rebuild any cache inside the notebook.

Design choices:

- one representative day for the figures: `2025-11-03`
- three days for the volatility table: `2025-11-03`, `2025-11-06`, `2025-11-20`
- only cached `1-second` `mid_price` series are used
- if a required cache file is missing, the notebook fails immediately with an explicit command to run

This keeps the workflow safe, fast, and suitable for a laptop.

## 1. Setup

Methodology:

- cached files contain only `timestamp` and `mid_price`
- mid-prices are already sampled at `1 second`
- annualized volatility is computed from `1-second` log-returns using

$$\sqrt{252 \times 9 \times 3600}$$

- missing caches are not built here; the helper script `build_light_midprice_cache.py` must be run manually

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CACHE_DIR = ROOT / "data/processed/price_dynamics_volatility_light"
PLOT_DAY = pd.Timestamp("2025-11-03").date()
BENCHMARK_DAYS = [
    pd.Timestamp("2025-11-03").date(),
    pd.Timestamp("2025-11-06").date(),
    pd.Timestamp("2025-11-20").date(),
]
MODEL_ORDER = ["Real", "QRU", "QR", "FTQR", "SAQR"]
SIM_MODELS = ["QRU", "QR", "FTQR", "SAQR"]
MODEL_COLORS = {
    "Real": "#111111",
    "QRU": "#4c78a8",
    "QR": "#f58518",
    "FTQR": "#54a24b",
    "SAQR": "#b279a2",
}
PERIODS = {
    "Full day": ("09:00:00", "18:00:00"),
    "Calm": ("10:00:00", "14:00:00"),
    "Active": ("15:00:00", "18:00:00"),
}
ANNUALIZATION_FACTOR = np.sqrt(252 * 9 * 3600)

def cache_path(model_name, day):
    return CACHE_DIR / f"{model_name.lower()}_{day}.parquet"

def build_hint(model_name, day, start_time, end_time):
    return (
        f"/Users/ulysse/miniconda3/envs/trading/bin/python build_light_midprice_cache.py "
        f"--model {model_name.lower()} --day {day} --start-time {start_time} --end-time {end_time}"
    )

def load_cached_mid(model_name, day, required_start, required_end):
    path = cache_path(model_name, day)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing cache file: {path.name}. Run: {build_hint(model_name, day, required_start, required_end)}"
        )
    df = pd.read_parquet(path)
    if "timestamp" in df.columns:
        df = df.set_index("timestamp")
    df.index = pd.to_datetime(df.index)
    if "mid_price" not in df.columns:
        raise ValueError(f"Cache file {path.name} is missing the mid_price column.")
    start = pd.Timestamp(f"{day} {required_start}", tz="Europe/Berlin")
    end = pd.Timestamp(f"{day} {required_end}", tz="Europe/Berlin")
    if df.index.min() > start or df.index.max() < end - pd.Timedelta(seconds=1):
        raise ValueError(
            f"Cache file {path.name} does not cover {required_start}–{required_end}. "
            f"Rebuild it with: {build_hint(model_name, day, required_start, required_end)}"
        )
    return df[["mid_price"]].sort_index()

def rolling_annualized_vol(series, window_seconds=600):
    returns = np.log(series).diff()
    return returns.rolling(window_seconds).std() * ANNUALIZATION_FACTOR

def period_volatility(df, start_time, end_time):
    window = df.between_time(start_time, end_time, inclusive="left")
    returns = np.log(window["mid_price"]).diff().dropna()
    if returns.empty:
        return np.nan
    return float(returns.std() * ANNUALIZATION_FACTOR)

manifest_rows = []
for day in BENCHMARK_DAYS:
    for model in MODEL_ORDER:
        manifest_rows.append({
            "date": pd.Timestamp(day),
            "model": model,
            "path": cache_path(model, day).name,
            "cached": cache_path(model, day).exists(),
        })
manifest_df = pd.DataFrame(manifest_rows)
display(manifest_df)
display(Markdown(
    "This notebook never computes missing series. If a required file is absent, it raises a clear error with the exact helper-script command to run."
))


## 2. Mid-Price Dynamics (1 Day)

Figure 9-style comparison over the representative window `10:00–12:30` on `2025-11-03`.

In [ ]:
plot_series = {}
for model in MODEL_ORDER:
    plot_series[model] = load_cached_mid(model, PLOT_DAY, "10:00:00", "12:30:00").loc[
        pd.Timestamp(f"{PLOT_DAY} 10:00:00", tz="Europe/Berlin"):pd.Timestamp(f"{PLOT_DAY} 12:30:00", tz="Europe/Berlin")
    ]

fig, ax = plt.subplots(figsize=(14, 5))
for model in MODEL_ORDER:
    ax.plot(plot_series[model].index, plot_series[model]["mid_price"], label=model, linewidth=1.8, color=MODEL_COLORS[model])
ax.set_title(f"Mid-Price Dynamics on {PLOT_DAY} (10:00–12:30)")
ax.set_xlabel("Time")
ax.set_ylabel("Mid-price")
ax.grid(alpha=0.2)
ax.legend(ncol=5, frameon=False)
fig.tight_layout()


## 3. Volatility (1 Day)

10-minute rolling annualized volatility from the same 1-second sampled mid-price series.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for model in MODEL_ORDER:
    vol = rolling_annualized_vol(plot_series[model]["mid_price"], window_seconds=600)
    ax.plot(vol.index, vol, label=model, linewidth=1.8, color=MODEL_COLORS[model])
ax.set_title(f"Rolling Annualized Volatility on {PLOT_DAY} (10-minute windows)")
ax.set_xlabel("Time")
ax.set_ylabel("Annualized volatility")
ax.grid(alpha=0.2)
ax.legend(ncol=5, frameon=False)
fig.tight_layout()

window_vol_df = pd.DataFrame(
    [{"model": model, "ann_vol": period_volatility(plot_series[model], "10:00:00", "12:30:00")} for model in MODEL_ORDER]
)
display(window_vol_df.style.format({"ann_vol": "{:.4f}"}))


## 4. Volatility Comparison Table (3 Days)

A light Table 2-style benchmark over three days only. The notebook loads one day at a time and stores only summary rows.

In [ ]:
rows = []
for day in BENCHMARK_DAYS:
    real_df = load_cached_mid("Real", day, "09:00:00", "18:00:00")
    day_models = {model: load_cached_mid(model, day, "09:00:00", "18:00:00") for model in SIM_MODELS}
    for period_name, (start_time, end_time) in PERIODS.items():
        sigma_real = period_volatility(real_df, start_time, end_time)
        for model in SIM_MODELS:
            sigma_sim = period_volatility(day_models[model], start_time, end_time)
            rows.append(
                {
                    "date": pd.Timestamp(day),
                    "period": period_name,
                    "model": model,
                    "sigma_real": sigma_real,
                    "sigma_sim": sigma_sim,
                    "relative_diff_pct": 100.0 * (sigma_sim - sigma_real) / sigma_real if sigma_real and not np.isnan(sigma_real) else np.nan,
                    "quadratic_error": (sigma_sim - sigma_real) ** 2 if pd.notna(sigma_sim) and pd.notna(sigma_real) else np.nan,
                }
            )

benchmark_df = pd.DataFrame(rows)
summary_table = (
    benchmark_df.groupby(["period", "model"], sort=False)
    .agg(
        avg_relative_diff_pct=("relative_diff_pct", "mean"),
        avg_quadratic_error=("quadratic_error", "mean"),
        sigma_real_mean=("sigma_real", "mean"),
        sigma_sim_mean=("sigma_sim", "mean"),
    )
    .reset_index()
)
summary_table["period"] = pd.Categorical(summary_table["period"], categories=list(PERIODS.keys()), ordered=True)
summary_table["model"] = pd.Categorical(summary_table["model"], categories=SIM_MODELS, ordered=True)
summary_table = summary_table.sort_values(["period", "model"]).reset_index(drop=True)

display(
    summary_table.style.format(
        {
            "avg_relative_diff_pct": "{:+.2f}",
            "avg_quadratic_error": "{:.6f}",
            "sigma_real_mean": "{:.4f}",
            "sigma_sim_mean": "{:.4f}",
        }
    )
)

best_model = summary_table.loc[summary_table.groupby("period")["avg_quadratic_error"].idxmin(), ["period", "model"]]
best_lines = "\n".join([f"- **{row.period}**: {row.model}" for row in best_model.itertuples(index=False)])
display(Markdown(
    f"""
### Interpretation

- Best model by average quadratic error in each period:
{best_lines}
- This is a deliberately reduced benchmark built for runtime safety and fast report generation.
"""
))


## 5. Limitations

This notebook is intentionally reduced:

- only one day is used for the figures,
- only three days are used for the table,
- only cached mid-price files are loaded,
- and all heavy computation is moved out of the notebook.

That makes the workflow much safer on a laptop while still preserving a representative comparison between Real, QRU, QR, FTQR, and SAQR.